In [0]:
import pyspark
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,DoubleType,DateType

In [0]:
DATA_PATH = "/Volumes/oag/bronze/raw/supply_chain_dataset.csv"
catalog_name = 'oag'

In [0]:
def define_schema():
    bronze_schema = StructType(
        [
            StructField("transaction_id", StringType(), True),
            StructField("transaction_date", DateType(), True),
            StructField("transaction_year", IntegerType(), True),
            StructField("transaction_quarter", IntegerType(), True),
            StructField("transaction_month", IntegerType(), True),
            StructField("product_name", StringType(), True),
            StructField("product_category", StringType(), True),
            StructField("quantity_unit", StringType(), True),
            StructField("supplier_name", StringType(), True),
            StructField("supplier_country", StringType(), True),
            StructField("supplier_reliability_score", DoubleType(), True),
            StructField("refinery_name", StringType(), True),
            StructField("destination_city", StringType(), True),
            StructField("transportation_mode", StringType(), True),
            StructField("ordered_quantity", DoubleType(), True),
            StructField("demand_quantity", DoubleType(), True),
            StructField("available_inventory", DoubleType(), True),
            StructField("unit_price_usd", DoubleType(), True),
            StructField("product_cost_usd", DoubleType(), True),
            StructField("transportation_cost_usd", DoubleType(), True),
            StructField("total_cost_usd", DoubleType(), True),
            StructField("expected_lead_time_days", IntegerType(), True),
            StructField("actual_lead_time_days", IntegerType(), True),
            StructField("delay_days", IntegerType(), True),
            StructField("is_delayed", IntegerType(), True),
            StructField("is_stockout", IntegerType(), True),
            StructField("quality_status", StringType(), True),
            StructField("quality_score", DoubleType(), True),
            StructField("disruption_type", StringType(), True),
            StructField("delivery_status",StringType(),True),
            StructField("ingestion_timestamp",StringType(),True),
            StructField("source_system",StringType(),True),
            StructField("operation_type",StringType(),True),
            ]
    )
    return bronze_schema

In [0]:
def create_dataframe():
    bronze_schema = define_schema()
    bronze_df = spark.read.csv(
        path=DATA_PATH,
        schema=bronze_schema,
        header=True,
        sep=","
    )
    bronze_df = bronze_df.withColumn(
        "ingested_at",
        F.current_timestamp()
    )
    return bronze_df

In [0]:
def create_table():
    bronze_df = create_dataframe()
    bronze_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{catalog_name}.bronze.bronze_table")
    return True


In [0]:
if create_table():
    print("Table Created Successfully")
else:
    print("Table Creation Unsuccessfull")

In [0]:
source_count = spark.read \
    .option("header", True) \
    .csv(DATA_PATH) \
    .count()

table_count = spark.table(f"{catalog_name}.bronze.bronze_table").count()

print("Source records :", source_count)
print("Table records  :", table_count)
print("Match          :", source_count == table_count)

In [0]:
df = spark.table(f"{catalog_name}.bronze.bronze_table")


In [0]:
df.printSchema()